In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import time
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from torch.amp import GradScaler, autocast

from utils.keypoint_dataset import WLASLKeypointDataset, NpyKeypointDataset

%load_ext autoreload 
%autoreload 2

In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

In [ ]:
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 64
EPOCHS      = 60           # 关键点模型收敛慢，给多一点
NUM_FRAMES  = 32
NUM_CLASSES = 300
INPUT_SIZE  = 126          # 2手 × 21点 × 3坐标
JSON_FILE   = ""
VIDEO_ROOT  = ""
CHECKPOINT  = ""
WORKERS     = 0

In [11]:
class KeypointLSTM(nn.Module):
    def __init__(self, input_size=126, hidden=512, num_layers=2,
                 num_classes=300, dropout=0.5):
        super().__init__()

        # 输入投影：先升维，让 LSTM 有足够表达空间
        self.input_proj = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True   # 双向，同时看过去和未来
        )

        # 双向所以 hidden*2
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, num_classes)
        )

        total = sum(p.numel() for p in self.parameters())
        print(f"[LSTM] Parameters: {total:,}")

    def forward(self, x):
        # x: [B, T, 126]
        x   = self.input_proj(x)              # [B, T, 256]
        out, (h, _) = self.lstm(x)            # out: [B, T, hidden*2]

        # 取最后时刻的前向+后向拼接，也可以用 mean pooling
        feat = out[:, -1, :]                  # [B, hidden*2]
        return self.classifier(feat)


In [12]:
def run_epoch(model, loader, optimizer, criterion, scaler,
              device, epoch, total_epochs, train=True):
    model.train() if train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    total_time = 0.0
    latencies  = []

    tag  = "Train" if train else "Val"
    pbar = tqdm(loader, desc=f"{tag} [{epoch+1}/{total_epochs}]", leave=False)

    for kp, labels in pbar:
        kp, labels = kp.to(device), labels.to(device)

        t0 = time.perf_counter()

        if train:
            optimizer.zero_grad()
            with autocast('cuda'):
                out  = model(kp)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                with autocast('cuda'):
                    out  = model(kp)
                    loss = criterion(out, labels)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1          = time.perf_counter()
            batch_time  = t1 - t0
            total_time += batch_time
            latencies.append(batch_time / kp.size(0) * 1000)

        total_loss += loss.item()
        _, pred = out.max(1)
        total   += labels.size(0)
        correct += pred.eq(labels).sum().item()
        pbar.set_postfix(loss=f"{loss.item():.3f}",
                         acc=f"{100.*correct/total:.1f}%")

    avg_loss = total_loss / len(loader)
    acc      = 100. * correct / total
    avg_lat  = float(np.mean(latencies)) if latencies else 0.0
    fps      = total / total_time        if total_time > 0 else 0.0

    if not train:
        print(f"Val Acc: {acc:.2f} | Latency: {avg_lat:.2f} ms/video | FPS: {fps:.1f}")

    return avg_loss, acc, avg_lat, fps

In [13]:
from collections import Counter

In [ ]:
if __name__ == "__main__":
    os.makedirs(os.path.dirname(CHECKPOINT), exist_ok=True)

    # To your NPY file
    NPY_ROOT = ""

    train_set = NpyKeypointDataset(JSON_FILE, NPY_ROOT, split='train')
    val_set   = NpyKeypointDataset(JSON_FILE, NPY_ROOT, split='val',
                                   label_map=train_set.action_to_idx)
    test_set  = NpyKeypointDataset(JSON_FILE, NPY_ROOT, split='test',
                                   label_map=train_set.action_to_idx)

    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=WORKERS, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=WORKERS, pin_memory=True)

    print(f"Device: {DEVICE}")

    # 💥 核心修复：直接读取字典的长度，数据集里有多少类，网络就输出多少维！
    ACTUAL_NUM_CLASSES = len(train_set.action_to_idx)
    print(f"✅ 真实存在的类别总数检测为: {ACTUAL_NUM_CLASSES} 类")


    labels = [s[1] for s in train_set.samples]
    cnt = Counter(labels)
    print(f"类别数: {len(cnt)}, 最多样本数: {max(cnt.values())}, 最少: {min(cnt.values())}, 平均: {sum(cnt.values())/len(cnt):.1f}")

    # 把 ACTUAL_NUM_CLASSES 传给模型
    model  = KeypointLSTM(INPUT_SIZE, hidden=128, num_layers=1,
                             num_classes=ACTUAL_NUM_CLASSES).to(DEVICE)
    
    # 学习率调低，配合更强的 weight_decay
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-2)
    
    # label_smoothing 加大（类别多、样本少时有用）
    criterion = nn.CrossEntropyLoss(label_smoothing=0.2)
                             
    # optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    # criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler    = GradScaler('cuda')

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        t0 = time.time()

        train_loss, train_acc, _, _ = run_epoch(
            model, train_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EPOCHS, train=True)
        val_loss, val_acc, val_lat, val_fps = run_epoch(
            model, val_loader, optimizer, criterion, scaler,
            DEVICE, epoch, EPOCHS, train=False)

        scheduler.step()
        duration = time.time() - t0

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | "
              f"{duration:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'label_map': train_set.action_to_idx,
            }, CHECKPOINT)
            print(f"✅ Best model saved (val acc {val_acc:.2f}%)")

    print("\n" + "="*50)
    ckpt = torch.load(CHECKPOINT)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_lat, test_fps = run_epoch(
        model, test_loader, None, criterion, scaler,
        DEVICE, 0, 1, train=False)
    print(f"Final Test Acc: {test_acc:.2f}% | "
          f"Latency: {test_lat:.2f} ms/video | FPS: {test_fps:.1f}")

[TRAIN] 成功加载 1897 个 .npy 特征文件
[VAL] 成功加载 446 个 .npy 特征文件
[TEST] 成功加载 317 个 .npy 特征文件
Device: cuda
✅ 真实存在的类别总数检测为: 300 类
类别数: 300, 最多样本数: 12, 最少: 1, 平均: 6.3
[LSTM] Parameters: 505,388


/home/haod6/.conda/envs/dlcv/lib/python3.13/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.5 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Train [1/60]:   0%|          | 0/30 [00:00<?, ?it/s]

Val [1/60]:   0%|          | 0/7 [00:00<?, ?it/s]

Val Acc: 0.67 | Latency: 0.07 ms/video | FPS: 13557.1
Epoch [01/60] Train Loss 6.0130 Acc 0.21% | Val Loss 5.7598 Acc 0.67% | 6.7s
✅ Best model saved (val acc 0.67%)


...

Train [29/60]:   0%|          | 0/30 [00:00<?, ?it/s]

Val [29/60]:   0%|          | 0/7 [00:00<?, ?it/s]

Val Acc: 1.35 | Latency: 0.04 ms/video | FPS: 23237.8
Epoch [29/60] Train Loss 5.2178 Acc 2.00% | Val Loss 5.3132 Acc 1.35% | 5.9s


Train [30/60]:   0%|          | 0/30 [00:00<?, ?it/s]

KeyboardInterrupt: 